## making my own gru model, using tokenizer of roberta then distillation training the custom model on roberta

In [1]:
import torch
import torch.nn as nn
from transformers.modeling_outputs import SequenceClassifierOutput


class GRUForSequenceClassification(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
        num_classes,
        pad_token_id=1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=pad_token_id
        )
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.1 if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        embedded = self.embedding(input_ids)
        outputs, _ = self.gru(embedded)

        if attention_mask is not None:
            mask_expanded = (
                attention_mask.unsqueeze(-1).expand(outputs.size()).float()
            )
            sum_outputs = torch.sum(outputs * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            pooled_output = sum_outputs / sum_mask
        else:
            pooled_output = outputs.mean(dim=1)

        logits = self.classifier(pooled_output)

        # Optional: Compute standard Cross-Entropy loss if hard labels are passed.
        # This allows standard HF Trainer evaluation metrics to work out-of-the-box.
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)

d:\anaconda\envs\nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
teacher_model_path = "final_peft_teacher_model"
model = AutoModelForSequenceClassification.from_pretrained(teacher_model_path)
tokenizer = AutoTokenizer.from_pretrained(teacher_model_path)
vocab_size = len(tokenizer)

Loading weights: 100%|██████████| 125/125 [00:00<00:00, 6249.41it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: final_peft_teacher_model
Key                                                                       | Status     | 
--------------------------------------------------------------------------+------------+-
roberta.encoder.layer.{0...11}.attention.self.query.base_layer.weight     | UNEXPECTED | 
roberta.encoder.layer.{0...11}.attention.self.value.base_layer.weight     | UNEXPECTED | 
roberta.encoder.layer.{0...11}.attention.self.key.base_layer.bias         | UNEXPECTED | 
roberta.encoder.layer.{0...11}.attention.self.value.base_layer.bias       | UNEXPECTED | 
classifier.original_module.dense.weight                                   | UNEXPECTED | 
roberta.encoder.layer.{0...11}.attention.self.value.lora_B.default.weight | UNEXPECTED | 
roberta.encoder.layer.{0...11}.attention.self.key.base_layer.weight       | UNEXPECTED | 
roberta.encoder.layer.{0...

In [3]:
print(vocab_size)

250002


In [4]:
pad_token_id = tokenizer.pad_token_id
print(f"Pad token ID: {pad_token_id}")

Pad token ID: 1


In [5]:
embedding_dim = 128
hidden_dim = 256
num_layers = 1 
num_classes = model.config.num_labels # Ensure student has same number of output classes as teacher

# 3. Initialize the model
student_gru = GRUForSequenceClassification(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    num_classes=num_classes,
    pad_token_id=pad_token_id
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student_gru = student_gru.to(device)

In [6]:
device

device(type='cuda')

In [7]:
from datasets import load_from_disk

dataset = load_from_disk("data/language_identification")
dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'text'],
        num_rows: 70000
    })
    validation: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
})

In [8]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128,  # Keep compact for faster recurrent processing
    )
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Standard Hugging Face datasets name the target column "label". 
# The Trainer expects it to be named "labels", otherwise it ignores the loss during evaluation!
if "label" in tokenized_datasets["train"].column_names:
    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

tokenized_datasets.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['labels', 'text', 'input_ids', 'attention_mask'],
        num_rows: 70000
    })
    validation: Dataset({
        features: ['labels', 'text', 'input_ids', 'attention_mask'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['labels', 'text', 'input_ids', 'attention_mask'],
        num_rows: 10000
    })
})

In [9]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(tokenized_datasets["train"], batch_size=32, shuffle=True)
eval_dataloader = DataLoader(tokenized_datasets["validation"], batch_size=32)
test_dataloader = DataLoader(tokenized_datasets["test"], batch_size=32)
print(f"len(train_dataloader): {len(train_dataloader)}")
print(f"len(eval_dataloader): {len(eval_dataloader)}")
print(f"len(test_dataloader): {len(test_dataloader)}")

len(train_dataloader): 2188
len(eval_dataloader): 313
len(test_dataloader): 313


In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments

class DistillationTrainer(Trainer):
    def __init__(self, teacher_model, distillation_alpha=0.5, temperature=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        student_device = next(self.model.parameters()).device
        self.teacher_model = teacher_model.to(self.model.device)
        self.teacher_model.eval()
        self.distillation_alpha = distillation_alpha
        self.temperature = temperature

    def compute_loss(self, model, inputs, return_outputs=False):
        # Student forward pass
        outputs_student = model(**inputs)
        student_logits = outputs_student.logits

        # Teacher forward pass
        with torch.no_grad():
            outputs_teacher = self.teacher_model(**inputs)
            teacher_logits = outputs_teacher.logits

        # Cross-Entropy Loss
        loss_ce = outputs_student.loss
        
        # Fallback if your custom model doesn't return loss natively
        if loss_ce is None and "labels" in inputs:
            loss_ce = F.cross_entropy(student_logits, inputs["labels"])

        # KL Divergence Distillation Loss
        loss_kd = nn.KLDivLoss(reduction="batchmean")(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1)
        ) * (self.temperature ** 2)

        # Combined Loss
        loss = (self.distillation_alpha * loss_ce) + ((1.0 - self.distillation_alpha) * loss_kd)

        return (loss, outputs_student) if return_outputs else loss

    # --- ADD THIS TO FIX THE EVAL_LOSS KEYERROR ---
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """Forces the trainer to compute our custom loss during the evaluation loop."""
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            # Grab the custom loss from our logic above
            loss, outputs = self.compute_loss(model, inputs, return_outputs=True)
            
        # Detach the loss tensor to prevent memory leaks during eval
        return (loss.detach(), None, None)

In [20]:
from transformers import PretrainedConfig


student_config = PretrainedConfig(
    vocab_size=tokenizer.vocab_size, 
    pad_token_id=tokenizer.pad_token_id,
    model_type="gru",
    hidden_size=256 
)

# 2. Attach the config and a dummy name to your in-memory model
student_gru.config = student_config
student_gru.name_or_path = "jupyter-notebook-gru-student"

In [21]:
from transformers import TrainingArguments, EarlyStoppingCallback


training_args = TrainingArguments(
    output_dir="./distilled_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    
    # Validation Strategy
    eval_strategy="epoch", # or "epoch"
    eval_steps=100,        # Evaluate every 100 steps
    save_strategy="epoch",
    save_steps=100,        # Save checkpoint every 100 steps
    load_best_model_at_end=True, # Recommended to keep the best version
    metric_for_best_model="eval_loss",
    
    push_to_hub=False, # Disable if you don't want to push to Hugging Face Hub
    hub_model_id=None,
    
    learning_rate=2e-5,
    num_train_epochs=50,
    logging_steps=10,
    report_to="none", # Or "wandb"/"tensorboard"
    remove_unused_columns=False, # Required so it doesn't remove input_ids
    fp16=torch.cuda.is_available() # Enable mixed precision if GPU is available
)

# Move the teacher model to the same device as the student model
# Standard PyTorch nn.Module does not have a .device property by default, so we use the global device variable
model = model.to(device)

trainer = DistillationTrainer(
    model=student_gru,
    teacher_model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    distillation_alpha=0.4, # 40% CE, 60% Teacher mimicry
    temperature=2.0,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

AttributeError: 'GRUForSequenceClassification' object has no attribute 'device'

In [22]:
trainer.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 